# Per-barcode sequence features

One row per printed feature, for the three layouts. The table is cached in `output/features_<layout>.parquet`; delete the files to recompute (about 25 minutes on 8 cores, dominated by the nearest-neighbour search).

| Group | Feature | Definition |
|---|---|---|
| Composition | `frac_A` ... `frac_T`, `GC`, `purine` | Fraction over the 36 nt |
| | `G_first18`, `G_last18` | G fraction of each 18-nt half (Dual: block identifier / spot code; Zipcodes: column / row code) |
| Motifs | `max_homopolymer`, `n_GG`, `n_GGG` | Longest run; overlapping GG and GGG counts |
| | `g4hunter` | G4Hunter score (runs of G scored +1 to +4, runs of C -1 to -4, averaged) |
| Synthesis | `n_cycles` | Cycles needed under left-most embedding in the A, C, G, T order |
| | `hamming_neighbours` | Mean Hamming distance between the feature's schedule and those of its 8 neighbours |
| | `stray_events` | Number of (cycle, neighbour) pairs in which a neighbour is illuminated while the feature is not, over all barcode cycles of the array |
| | `stray_cycles` | Number of cycles in which at least one neighbour is illuminated while the feature is not |
| Structure | `mfe_barcode`, `mfe_probe` | Minimum free energy (kcal/mol, ViennaRNA, DNA parameters of Mathews 2004) of the barcode alone, and of Read 1 handle + N10 + barcode + Read 2 handle |
| Decoding | `nn_dist` | Levenshtein distance to the nearest other entry of the combined whitelist (real + decoys) |
| | `n_within_7` | Number of whitelist entries within distance 7 |
| | `kmer4_mean_log_freq`, `kmer4_min_freq` | Library frequency of the barcode's 4-mers (number of whitelist entries containing each 4-mer) |

Assumption: the arrays were synthesized with left-most embedding of the barcodes in the repeating A, C, G, T order, all barcodes starting in the same cycle after the UMI.

In [1]:
from concurrent.futures import ProcessPoolExecutor
from itertools import product
import time

import numpy as np
import pandas as pd
from rapidfuzz.distance import Levenshtein
from rapidfuzz.process import cdist

from common import ARRAYS, OUT, ROWS, COLS, SEG, BC_LEN, fold_batch, load_layout, load_whitelist, leftmost_schedule

NN_CUTOFF = 20


def g4hunter(seq):
    scores = np.zeros(len(seq))
    i = 0
    while i < len(seq):
        j = i
        while j < len(seq) and seq[j] == seq[i]:
            j += 1
        if seq[i] in "GC":
            scores[i:j] = min(j - i, 4) * (1 if seq[i] == "G" else -1)
        i = j
    return scores.mean()


def max_run(seq):
    best = run = 1
    for a, b in zip(seq, seq[1:]):
        run = run + 1 if a == b else 1
        best = max(best, run)
    return best


def count_overlapping(seq, motif):
    return sum(seq.startswith(motif, i) for i in range(len(seq) - len(motif) + 1))


def schedule_features(lay):
    cycles = [leftmost_schedule(s) for s in lay.barcode]
    T = max(c[-1] for c in cycles) + 1
    sched = np.zeros((ROWS, COLS, T), dtype=bool)
    for (r, c), cyc in zip(zip(lay.row, lay.col), cycles):
        sched[r, c, cyc] = True
    pad = np.zeros((ROWS + 2, COLS + 2, T), dtype=bool)
    pad[1:-1, 1:-1] = sched
    present = np.zeros((ROWS + 2, COLS + 2), dtype=bool)
    present[1:-1, 1:-1] = True
    ham = np.zeros((ROWS, COLS))
    n_nb = np.zeros((ROWS, COLS))
    events = np.zeros((ROWS, COLS))
    any_lit = np.zeros((ROWS, COLS, T), dtype=bool)
    for dr, dc in product((-1, 0, 1), repeat=2):
        if dr == dc == 0:
            continue
        nb = pad[1 + dr:ROWS + 1 + dr, 1 + dc:COLS + 1 + dc]
        ok = present[1 + dr:ROWS + 1 + dr, 1 + dc:COLS + 1 + dc]
        ham += ok * (nb ^ sched).sum(axis=2)
        n_nb += ok
        lit_while_dark = nb & ~sched
        events += lit_while_dark.sum(axis=2)
        any_lit |= lit_while_dark
    feats = pd.DataFrame({
        "n_cycles": [c[-1] + 1 for c in cycles],
        "hamming_neighbours": (ham / n_nb)[lay.row, lay.col],
        "stray_events": events[lay.row, lay.col],
        "stray_cycles": any_lit.sum(axis=2)[lay.row, lay.col],
    })
    return feats, T


def decoding_features(name, lay):
    real, decoys = load_whitelist(name)
    wl = real + decoys
    index = {s: i for i, s in enumerate(wl)}
    nn, within = np.empty(len(lay), dtype=int), np.empty(len(lay), dtype=int)
    chunk = 2000
    for start in range(0, len(lay), chunk):
        q = list(lay.barcode[start:start + chunk])
        d = cdist(q, wl, scorer=Levenshtein.distance, dtype=np.uint8, workers=-1, score_cutoff=NN_CUTOFF)
        d[np.arange(len(q)), [index[s] for s in q]] = 255
        nn[start:start + chunk] = d.min(axis=1)
        within[start:start + chunk] = (d <= 7).sum(axis=1)
    kfreq = {}
    for s in wl:
        for k in {s[i:i + 4] for i in range(len(s) - 3)}:
            kfreq[k] = kfreq.get(k, 0) + 1
    kmer_mean, kmer_min = [], []
    for s in lay.barcode:
        f = np.array([kfreq[s[i:i + 4]] for i in range(len(s) - 3)])
        kmer_mean.append(np.log10(f).mean())
        kmer_min.append(f.min())
    return pd.DataFrame({"nn_dist": nn, "n_within_7": within,
                         "kmer4_mean_log_freq": kmer_mean, "kmer4_min_freq": kmer_min})


def build(name):
    lay = load_layout(name)
    s = lay.barcode
    feats = pd.DataFrame({"row": lay.row, "col": lay.col, "barcode": s})
    for b in "ACGT":
        feats[f"frac_{b}"] = s.str.count(b) / BC_LEN
    feats["GC"] = feats.frac_G + feats.frac_C
    feats["purine"] = feats.frac_A + feats.frac_G
    feats["G_first18"] = s.str[:SEG].str.count("G") / SEG
    feats["G_last18"] = s.str[SEG:].str.count("G") / SEG
    feats["max_homopolymer"] = s.map(max_run)
    feats["n_GG"] = s.map(lambda x: count_overlapping(x, "GG"))
    feats["n_GGG"] = s.map(lambda x: count_overlapping(x, "GGG"))
    feats["g4hunter"] = s.map(g4hunter)
    sched, T = schedule_features(lay)
    feats = pd.concat([feats, sched], axis=1)
    t = time.time()
    batches = [list(s[i:i + 2000]) for i in range(0, len(s), 2000)]
    with ProcessPoolExecutor() as ex:
        folds = [x for batch in ex.map(fold_batch, batches) for x in batch]
    feats["mfe_barcode"] = [f[0] for f in folds]
    feats["mfe_probe"] = [f[1] for f in folds]
    print(f"{name}: folding {time.time() - t:.0f}s, {T} barcode cycles")
    t = time.time()
    feats = pd.concat([feats, decoding_features(name, lay)], axis=1)
    print(f"{name}: nearest-neighbour search {time.time() - t:.0f}s")
    feats["edge_dist"] = np.minimum.reduce([feats.row, feats.col, ROWS - 1 - feats.row, COLS - 1 - feats.col])
    return feats


features = {}
for name, cfg in ARRAYS.items():
    path = OUT / f"features_{cfg['key']}.parquet"
    if not path.exists():
        build(name).to_parquet(path)
    features[name] = pd.read_parquet(path)

Dual barcodes: folding 100s, 110 barcode cycles
Dual barcodes: nearest-neighbour search 363s
Greedy + 2-OPT: folding 105s, 109 barcode cycles
Greedy + 2-OPT: nearest-neighbour search 367s
Zipcodes: folding 104s, 118 barcode cycles
Zipcodes: nearest-neighbour search 328s


## Summary per layout

In [2]:
FEATURES = ["frac_A", "frac_C", "frac_G", "frac_T", "GC", "G_first18", "G_last18", "max_homopolymer", "n_GGG",
            "g4hunter", "n_cycles", "hamming_neighbours", "stray_events", "stray_cycles", "mfe_barcode", "mfe_probe",
            "nn_dist", "n_within_7", "kmer4_mean_log_freq", "kmer4_min_freq"]
summary = pd.concat({name: df[FEATURES].describe().T[["mean", "std", "min", "50%", "max"]]
                     for name, df in features.items()}, axis=1).round(3)
summary.to_csv(OUT / "feature_summary.csv")
summary

Dual barcodes                                          \
                             mean       std      min       50%        max   
frac_A                      0.252     0.064    0.000     0.250      0.528   
frac_C                      0.287     0.063    0.056     0.278      0.528   
frac_G                      0.215     0.051    0.028     0.222      0.417   
frac_T                      0.246     0.065    0.028     0.250      0.528   
GC                          0.502     0.067    0.250     0.500      0.806   
G_first18                   0.215     0.072    0.000     0.222      0.444   
G_last18                    0.215     0.072    0.000     0.222      0.500   
max_homopolymer             2.611     0.538    1.000     3.000      6.000   
n_GGG                       0.088     0.298    0.000     0.000      4.000   
g4hunter                   -0.137     0.195   -1.056    -0.139      0.667   
n_cycles                   84.950     5.959   59.000    85.000    110.000   
hamming_neighbours         28.675     5.054   16.400    29.250     43.500   
stray_events              114.160    20.959   34.000   117.000    174.000   
stray_cycles               39.135     7.939   19.000    40.000     64.000   
mfe_barcode                -2.241     1.626  -14.300    -2.000      0.000   
mfe_probe                 -23.718     1.869  -37.000   -23.500    -21.140   
nn_dist                     7.931     0.856    3.000     8.000     11.000   
n_within_7                  0.319     0.568    0.000     0.000      5.000   
kmer4_mean_log_freq         4.227     0.040    3.997     4.229      4.375   
kmer4_min_freq           7745.357  2496.033  238.000  7686.000  17265.000   

                    Greedy + 2-OPT                                           \
                              mean       std       min       50%        max   
frac_A                       0.252     0.062     0.000     0.250      0.528   
frac_C                       0.288     0.060     0.083     0.278      0.583   
frac_G                       0.215     0.035     0.000     0.222      0.250   
frac_T                       0.246     0.063     0.000     0.250      0.500   
GC                           0.502     0.060     0.306     0.500      0.806   
G_first18                    0.215     0.072     0.000     0.222      0.500   
G_last18                     0.214     0.072     0.000     0.222      0.500   
max_homopolymer              2.598     0.491     1.000     3.000      3.000   
n_GGG                        0.081     0.275     0.000     0.000      2.000   
g4hunter                    -0.139     0.171    -1.000    -0.111      0.361   
n_cycles                    84.709     5.593    61.000    85.000    109.000   
hamming_neighbours          29.213     2.204    20.000    29.000     49.333   
stray_events               116.229     9.908    35.000   116.000    170.000   
stray_cycles                44.282     4.084    22.000    44.000     63.000   
mfe_barcode                 -2.205     1.522   -14.600    -2.000      0.000   
mfe_probe                  -23.673     1.746   -35.500   -23.500    -21.140   
nn_dist                     11.525     0.672     7.000    12.000     13.000   
n_within_7                   0.000     0.008     0.000     0.000      1.000   
kmer4_mean_log_freq          4.231     0.032     4.123     4.229      4.396   
kmer4_min_freq            7327.777  2239.740  2592.000  7129.000  16539.000   

                     Zipcodes                                            
                         mean       std       min        50%        max  
frac_A                  0.252     0.071     0.000      0.250      0.556  
frac_C                  0.260     0.073     0.056      0.250      0.556  
frac_G                  0.233     0.069     0.000      0.222      0.528  
frac_T                  0.255     0.078     0.000      0.250      0.611  
GC                      0.493     0.084     0.111      0.500      0.750  
G_first18               0.250     0.101     0.000      0.222      0.

Correlation between features (Spearman), pooled over layouts, to identify redundant features before modelling.

In [3]:
pooled = pd.concat(features.values())
corr = pooled[FEATURES].corr(method="spearman").round(2)
corr.to_csv(OUT / "feature_correlation.csv")
corr

,frac_A,frac_C,frac_G,frac_T,GC,G_first18,G_last18,max_homopolymer,n_GGG,g4hunter,n_cycles,hamming_neighbours,stray_events,stray_cycles,mfe_barcode,mfe_probe,nn_dist,n_within_7,kmer4_mean_log_freq,kmer4_min_freq
frac_A,1.00,-0.30,-0.25,-0.43,-0.49,-0.17,-0.16,-0.01,-0.09,0.11,-0.07,-0.04,-0.04,-0.03,0.16,0.16,0.00,-0.01,0.04,0.01
frac_C,-0.30,1.00,-0.31,-0.41,0.69,-0.23,-0.17,-0.04,-0.12,-0.77,0.00,-0.04,-0.03,0.07,-0.08,-0.02,0.15,-0.16,0.37,0.01
frac_G,-0.25,-0.31,1.00,-0.19,0.42,0.65,0.61,-0.02,0.31,0.61,0.04,0.08,0.07,-0.02,-0.36,-0.40,-0.09,0.11,-0.47,-0.16
frac_T,-0.43,-0.41,-0.19,1.00,-0.53,-0.11,-0.15,0.03,-0.06,0.16,0.01,0.01,0.01,-0.01,0.21,0.17,-0.04,0.05,-0.03,0.08
GC,-0.49,0.69,0.42,-0.53,1.00,0.26,0.29,-0.03,0.15,-0.26,0.06,0.03,0.03,0.03,-0.35,-0.32,0.03,-0.03,-0.01,-0.08
G_first18,-0.17,-0.23,0.65,-0.11,0.26,1.00,-0.15,0.02,0.24,0.42,0.07,0.05,0.05,-0.04,-0.23,-0.25,-0.14,0.16,-0.33,-0.07
G_last18,-0.16,-0.17,0.61,-0.15,0.29,-0.15,1.00,-0.06,0.16,0.37,-0.02,0.04,0.04,-0.00,-0.24,-0.28,0.01,-0.02,-0.26,-0.13
max_homopolymer,-0.01,-0.04,-0.02,0.03,-0.03,0.02,-0.06,1.00,0.30,-0.07,0.44,0.14,0.14,-0.05,0.08,0.09,-0.33,0.36,-0.35,0.03
n_GGG,-0.09,-0.12,0.31,-0.06,0.15,0.24,0.16,0.30,1.00,0.39,0.23,0.07,0.07,-0.06,-0.09,-0.09,-0.19,0.22,-0.36,-0.14
g4hunter,0.11,-0.77,0.61,0.16,-0.26,0.42,0.37,-0.07,0.39,1.00,-0.04,0.05,0.04,-0.05,-0.13,-0.19,-0.12,0.13,-0.41,-0.19
